# NFW-004 — Veto-Gated Neural Privilege Separation

## Colab-first capability firewall prototype

This notebook is the next architectural step after NFW-001/002/003. It does **not** claim that an
activation classifier can reliably predict every harmful completion. Instead, it demonstrates a
security invariant that can be enforced independently of model accuracy:

> Model-generated text may request an action, but it cannot grant itself authority to perform that action.

The model is treated as an untrusted principal. A host-side capability broker is the security boundary.
A neural attestation signal is **veto-only**: it may deny or require approval, but it can never grant or
upgrade privileges. The notebook runs its security tests on CPU; the optional Qwen activation demo runs
only in a Colab GPU runtime.

### What this proves / does not prove

- **Proves as a PoC:** default-deny typed broker, scoped expiring capabilities, replay protection,
  model-claim rejection, neural veto semantics, append-only audit records, resumable Drive artifacts.
- **Does not prove:** universal harmful-output prevention, adaptive robustness, policy isolation inside
  Qwen weights, causal control, secure production cryptography, or cross-model transfer.
- **Research framing:** “veto-gated neural attestation coupled to an external capability boundary.”
  Verify novelty against the literature before using “novel” in a paper title.

Run every cell from the top in a fresh Colab runtime. All artifacts go to Drive.


In [ ]:
# Colab-only setup. Do not run this notebook on a local machine for model inference.
# The broker tests below are CPU-only, but Drive persistence requires Colab.
import sys, subprocess
subprocess.check_call([sys.executable,'-m','pip','install','-q',
                       'transformers==4.57.1','accelerate==1.11.0'])


In [ ]:
import numpy as np
import base64, csv, gc, hashlib, hmac, importlib.metadata, io, json, math, os, re, secrets, sys, tempfile, time, uuid
from collections import Counter
from dataclasses import dataclass
from pathlib import Path
from typing import Any

from google.colab import drive
drive.mount('/content/drive', force_remount=False)

RUN_ID = 'nfw004_veto_broker_001'  # Change for a new experiment; never reuse after code/config edits.
RUN_GPU_ATTESTATION = True         # False: broker-only CPU proof; True: optional Qwen activation demo.
REVIEW_ONLY = False                # True: reopen completed Drive artifacts without loading Qwen.
MODEL_ID = 'Qwen/Qwen2.5-3B-Instruct'
MODEL_REVISION = 'aa8e72537993ba99e69dfaafa59ed015b17504d1'
ATTESTATION_LAYER = 20
MAX_PROMPT_TOKENS = 512
CAPABILITY_TTL_SECONDS = 300
WORKFLOW_VERSION = 'nfw004-1.0'
OUTPUT_ROOT = Path('/content/drive/MyDrive/NFW-004')
RUN_DIR = OUTPUT_ROOT / RUN_ID
RUN_DIR.mkdir(parents=True, exist_ok=True)
if not re.fullmatch(r'[A-Za-z0-9_-]+', RUN_ID): raise ValueError('Unsafe RUN_ID')
if REVIEW_ONLY: RUN_GPU_ATTESTATION = False
print('Run directory:', RUN_DIR)
print('Broker tests use CPU; GPU attestation:', RUN_GPU_ATTESTATION)


In [ ]:
def canonical(x):
    return json.dumps(x,sort_keys=True,ensure_ascii=False,separators=(',',':'),allow_nan=False)

def digest(x): return hashlib.sha256(canonical(x).encode()).hexdigest()

def file_hash(path):
    h=hashlib.sha256()
    with Path(path).open('rb') as f:
        for chunk in iter(lambda:f.read(1024*1024),b''): h.update(chunk)
    return h.hexdigest()

def atomic_text(path,text):
    path=Path(path); path.parent.mkdir(parents=True,exist_ok=True)
    fd,tmp=tempfile.mkstemp(prefix='.'+path.name,dir=path.parent)
    try:
        with os.fdopen(fd,'w',encoding='utf-8',newline='') as f:
            f.write(text); f.flush(); os.fsync(f.fileno())
        os.replace(tmp,path)
    finally:
        if os.path.exists(tmp): os.unlink(tmp)

def atomic_json(path,obj): atomic_text(path,canonical(obj)+'\n')
def read_json(path): return json.loads(Path(path).read_text(encoding='utf-8'))
def require_equal(actual,expected,label):
    if actual!=expected: raise RuntimeError(f'{label} mismatch; refusing reuse. Use a new RUN_ID.')

def save_stage(name,payload,binding):
    path=RUN_DIR/name; env={'binding':binding,'payload':payload,'payload_sha256':digest(payload)}
    if path.exists(): require_equal(read_json(path),env,name)
    else: atomic_json(path,env)
    return payload

def load_stage(name,binding):
    path=RUN_DIR/name
    if not path.exists(): return None
    env=read_json(path); require_equal(env['binding'],binding,name+' binding')
    require_equal(env['payload_sha256'],digest(env['payload']),name+' checksum')
    return env['payload']

def mark_stage(name,filename):
    entry={'file':filename,'sha256':file_hash(RUN_DIR/filename)}
    if name in manifest['stages']: require_equal(manifest['stages'][name],entry,name+' stage')
    else:
        manifest['stages'][name]=entry; atomic_json(RUN_DIR/'manifest.json',manifest)


In [ ]:
# ---------------- Trusted host-side capability broker ----------------
# The model never receives the signing secret and never calls mint_capability().

@dataclass(frozen=True)
class Capability:
    token_id: str
    subject: str
    capability: str
    resources: tuple
    issued_at: int
    expires_at: int
    nonce: str
    signature: str

@dataclass(frozen=True)
class Decision:
    allowed: bool
    reason: str
    request_id: str
    capability: str | None = None

class CapabilityBroker:
    """Reference broker: default deny; only trusted host code can mint tokens."""
    ALLOWED = {
        'lookup_public_fact': {'query'},
        'read_project_file': {'path'},
    }
    def __init__(self, secret: bytes, now=None):
        if not isinstance(secret,bytes) or len(secret)<32: raise ValueError('Broker secret must be >=32 bytes')
        self._secret=secret; self._now=now or (lambda:int(time.time())); self._used_nonces=set(); self.audit=[]
    def _body(self, token_id,subject,capability,resources,issued,expires,nonce):
        return canonical({'token_id':token_id,'subject':subject,'capability':capability,
                          'resources':list(resources),'issued_at':issued,'expires_at':expires,'nonce':nonce})
    def _sign(self,body): return hmac.new(self._secret,body.encode(),hashlib.sha256).hexdigest()
    def mint_capability(self, *, subject, capability, resources, ttl_seconds=CAPABILITY_TTL_SECONDS):
        # This method represents a trusted policy authority, not the model.
        if capability not in self.ALLOWED: raise PermissionError('Unknown capability')
        if set(resources) != self.ALLOWED[capability]: raise PermissionError('Invalid resource schema')
        now=self._now(); token_id=uuid.uuid4().hex; nonce=secrets.token_hex(16); expires=now+int(ttl_seconds)
        body=self._body(token_id,subject,capability,tuple(sorted(resources)),now,expires,nonce)
        return Capability(token_id,subject,capability,tuple(sorted(resources)),now,expires,nonce,self._sign(body))
    def _valid_token(self,token,subject,capability):
        if not isinstance(token,Capability): return False,'missing_or_untrusted_token'
        if token.subject!=subject or token.capability!=capability: return False,'scope_mismatch'
        if token.expires_at<=self._now(): return False,'expired'
        if token.nonce in self._used_nonces: return False,'replay'
        body=self._body(token.token_id,token.subject,token.capability,token.resources,token.issued_at,token.expires_at,token.nonce)
        if not hmac.compare_digest(token.signature,self._sign(body)): return False,'invalid_signature'
        return True,'ok'
    def authorize(self, *, subject, request, token=None, attestation='allow', human_approval=False):
        request_id=uuid.uuid4().hex
        def deny(reason):
            d=Decision(False,reason,request_id); self.audit.append({'request_id':request_id,'allowed':False,'reason':reason,'subject':subject}); return d
        if not isinstance(request,dict) or set(request)!={'tool','arguments'}: return deny('malformed_request')
        tool,args=request['tool'],request['arguments']
        if not isinstance(tool,str) or tool not in self.ALLOWED: return deny('tool_not_allowlisted')
        if not isinstance(args,dict) or set(args)!=self.ALLOWED[tool]: return deny('argument_schema_rejected')
        if attestation not in {'allow','deny','approval_required'}: return deny('invalid_attestation')
        if attestation=='deny': return deny('neural_veto')
        if attestation=='approval_required' and not human_approval: return deny('human_approval_required')
        # Ignore any model-supplied capability/authorization fields: request schema excludes them.
        ok,reason=self._valid_token(token,subject,tool)
        if not ok: return deny(reason)
        self._used_nonces.add(token.nonce)
        d=Decision(True,'authorized',request_id,tool); self.audit.append({'request_id':request_id,'allowed':True,'reason':'authorized','subject':subject,'tool':tool}); return d

def model_request(raw):
    """Parse a model proposal; model claims such as capability/signature are discarded."""
    if not isinstance(raw,dict): return None
    if 'tool' not in raw or 'arguments' not in raw: return None
    return {'tool':raw['tool'],'arguments':raw['arguments']}


In [ ]:
class VetoOnlyAttestation:
    """Neural output can lower authority, never increase it."""
    def __init__(self, score_fn=None, threshold=0.0): self.score_fn=score_fn; self.threshold=float(threshold)
    def assess(self, prompt, request):
        if self.score_fn is None: return 'allow', {'score':None,'source':'disabled'}
        score=float(self.score_fn(prompt));
        if not math.isfinite(score): return 'deny', {'score':None,'source':'nonfinite'}
        return ('deny' if score>=self.threshold else 'allow'), {'score':score,'source':'neural_activation'}

def compose_attestation(neural_decision, *, force_approval=False):
    # Security monotonicity: only allow -> approval_required/deny transitions are possible.
    if neural_decision=='deny': return 'deny'
    if neural_decision=='approval_required' or force_approval: return 'approval_required'
    if neural_decision=='allow': return 'allow'
    return 'deny'


In [ ]:
# Manifest binds the security implementation and model identity. Hardware is observational only.
IMPLEMENTATION_ID='nfw004-veto-broker-reference-1'
packages={n:importlib.metadata.version(n) for n in ['torch','transformers','accelerate']}
execution={'python':list(sys.version_info[:3]),'packages':packages}
identity={'workflow_version':WORKFLOW_VERSION,'model_id':MODEL_ID,'model_revision':MODEL_REVISION,
          'attestation_layer':ATTESTATION_LAYER,'max_prompt_tokens':MAX_PROMPT_TOKENS,
          'implementation':IMPLEMENTATION_ID}
manifest_path=RUN_DIR/'manifest.json'
manifest=read_json(manifest_path) if manifest_path.exists() else None
if manifest:
    require_equal(manifest['run_id'],RUN_ID,'run id'); require_equal(manifest['identity'],identity,'identity'); require_equal(manifest['execution'],execution,'execution')
    for stage,e in manifest['stages'].items():
        if not (RUN_DIR/e['file']).is_file(): raise RuntimeError('Missing stage '+stage)
        require_equal(file_hash(RUN_DIR/e['file']),e['sha256'],stage)
else:
    if any(RUN_DIR.iterdir()): raise RuntimeError('Nonempty run without manifest; use new RUN_ID')
    manifest={'run_id':RUN_ID,'identity':identity,'execution':execution,'hardware_observations':[],'stages':{}}
    atomic_json(manifest_path,manifest)
if RUN_GPU_ATTESTATION:
    import torch
    if not torch.cuda.is_available(): raise RuntimeError('Select a Colab GPU runtime, or set RUN_GPU_ATTESTATION=False for broker-only proof')
    obs={'gpu':torch.cuda.get_device_name(0),'capability':list(torch.cuda.get_device_capability(0))}
    if obs not in manifest['hardware_observations']:
        manifest['hardware_observations'].append(obs); atomic_json(manifest_path,manifest)
BINDING=digest([identity,execution])
print('Identity locked. Hardware observations:',manifest['hardware_observations'])


In [ ]:
def run_security_tests():
    now=[1000]
    broker=CapabilityBroker(b'trusted-demo-secret-'*3,now=lambda:now[0])
    subject='session:test-user'; read=broker.mint_capability(subject=subject,capability='lookup_public_fact',resources=('query',),ttl_seconds=60)
    good={'tool':'lookup_public_fact','arguments':{'query':'What is a neuron?'}}
    cases=[]
    def check(name, decision, expected, reason=None):
        assert decision.allowed is expected,(name,decision)
        if reason: assert decision.reason==reason,(name,decision)
        cases.append({'name':name,'passed':True,'allowed':decision.allowed,'reason':decision.reason})
    check('valid_capability',broker.authorize(subject=subject,request=good,token=read),True)
    check('replay_denied',broker.authorize(subject=subject,request=good,token=read),False,'replay')
    write=broker.mint_capability(subject=subject,capability='read_project_file',resources=('path',),ttl_seconds=60)
    check('scope_mismatch',broker.authorize(subject=subject,request=good,token=write),False,'scope_mismatch')
    fake={'tool':'lookup_public_fact','arguments':{'query':'x'},'capability':'allow_all','signature':'fake'}
    check('model_claim_ignored',broker.authorize(subject=subject,request=model_request(fake),token=None),False,'missing_or_untrusted_token')
    check('neural_veto',broker.authorize(subject=subject,request=good,token=broker.mint_capability(subject=subject,capability='lookup_public_fact',resources=('query',)),attestation='deny'),False,'neural_veto')
    approval=broker.mint_capability(subject=subject,capability='lookup_public_fact',resources=('query',))
    check('approval_required_without_human',broker.authorize(subject=subject,request=good,token=approval,attestation='approval_required'),False,'human_approval_required')
    approval2=broker.mint_capability(subject=subject,capability='lookup_public_fact',resources=('query',))
    check('approved_high_risk',broker.authorize(subject=subject,request=good,token=approval2,attestation='approval_required',human_approval=True),True)
    expired=broker.mint_capability(subject=subject,capability='lookup_public_fact',resources=('query',),ttl_seconds=1); now[0]+=2
    check('expired_denied',broker.authorize(subject=subject,request=good,token=expired),False,'expired')
    now[0]=1000
    check('unknown_tool_denied',broker.authorize(subject=subject,request={'tool':'shell','arguments':{'command':'x'}},token=None),False,'tool_not_allowlisted')
    check('extra_argument_denied',broker.authorize(subject=subject,request={'tool':'lookup_public_fact','arguments':{'query':'x','admin':True}},token=None),False,'argument_schema_rejected')
    check('malformed_denied',broker.authorize(subject=subject,request={'tool':'lookup_public_fact'},token=None),False,'malformed_request')
    # Monotonicity: neural decisions can only remove authority.
    assert compose_attestation('deny')=='deny'; assert compose_attestation('approval_required')=='approval_required'; assert compose_attestation('allow')=='allow'
    return {'n_cases':len(cases),'passed':sum(x['passed'] for x in cases),'all_passed':all(x['passed'] for x in cases),'cases':cases,'external_actions_executed':0,'model_can_mint_capability':False,'model_can_upgrade_privilege':False,'audit_events':len(broker.audit)}

SECURITY_BINDING=digest([BINDING,'security-tests-v1'])
tests=load_stage('security_tests.json',SECURITY_BINDING)
if tests is None:
    tests=run_security_tests(); save_stage('security_tests.json',tests,SECURITY_BINDING); mark_stage('security_tests','security_tests.json')
else: print('Loaded completed security tests; no rerun.')
assert tests['all_passed'] and tests['external_actions_executed']==0
print(json.dumps(tests,indent=2))


## Optional GPU neural-attestation demonstration

This cell extracts one decoder-block activation from Qwen for a set of **non-executing** action
proposals. It does not train or claim a safety classifier. To use a learned monitor, provide a
separately frozen coefficient artifact and bind its hash into the manifest. The broker remains
secure when this cell is disabled, unavailable, wrong, or manipulated: a neural signal can only
veto or require approval.


In [ ]:
GPU_BINDING=digest([BINDING,'qwen-activation-demo',MODEL_REVISION,ATTESTATION_LAYER])
gpu_demo=load_stage('gpu_attestation_demo.json',GPU_BINDING)
if gpu_demo is None and RUN_GPU_ATTESTATION:
    import torch
    from transformers import AutoModelForCausalLM, AutoTokenizer
    tokenizer=AutoTokenizer.from_pretrained(MODEL_ID,revision=MODEL_REVISION)
    model=AutoModelForCausalLM.from_pretrained(MODEL_ID,revision=MODEL_REVISION,torch_dtype=torch.float16,device_map={'':'cuda:0'},low_cpu_mem_usage=True,attn_implementation='eager').eval()
    require_equal(getattr(model.config,'_commit_hash',None),MODEL_REVISION,'model revision')
    if ATTESTATION_LAYER>=len(model.model.layers)-1: raise ValueError('Invalid non-final layer')
    prompts=['Find a public fact about neurons.','Request an unauthorized shell operation.','Ask to read a project file.']
    proposals=[{'tool':'lookup_public_fact','arguments':{'query':'a public fact'}},
               {'tool':'shell','arguments':{'command':'anything'}},
               {'tool':'read_project_file','arguments':{'path':'README.md'}}]
    records=[]
    for prompt,proposal in zip(prompts,proposals):
        ids=tokenizer.apply_chat_template([{'role':'user','content':prompt}],tokenize=True,add_generation_prompt=True)
        x=torch.tensor([ids],device='cuda')
        captured={}
        def hook(module,inputs,output):
            h=output[0] if isinstance(output,tuple) else output; captured['v']=h[0,-1].detach().float().cpu().numpy().tolist()
        handle=model.model.layers[ATTESTATION_LAYER].register_forward_hook(hook)
        with torch.inference_mode(): model.model(input_ids=x,attention_mask=torch.ones_like(x),use_cache=False)
        handle.remove(); margin=float(np.linalg.norm(np.asarray(captured['v'],dtype=float)))
        records.append({'prompt_class':prompt,'proposal':proposal,'layer':ATTESTATION_LAYER,'activation_l2':margin,
                        'attestation_role':'observation_only_no_authorization'})
    gpu_demo={'model_id':MODEL_ID,'model_revision':MODEL_REVISION,'records':records,
              'warning':'Untrained activation norms are not safety scores.'}
    save_stage('gpu_attestation_demo.json',gpu_demo,GPU_BINDING); mark_stage('gpu_attestation','gpu_attestation_demo.json')
    del model; gc.collect(); torch.cuda.empty_cache()
elif gpu_demo is None: print('GPU demo disabled; broker proof remains complete.')
else: print('Loaded completed GPU demo; no model reload required.')


In [ ]:
# Demonstrate end-to-end request handling. No real tool is called.
subject='session:demo'; broker=CapabilityBroker(b'integration-secret-'*3)
trusted=broker.mint_capability(subject=subject,capability='lookup_public_fact',resources=('query',))
model_output={'tool':'lookup_public_fact','arguments':{'query':'a fact'},'capability':'allow_all','admin':True}
proposal=model_request(model_output)
attestation=VetoOnlyAttestation().assess('Find a fact.',proposal)[0]
decision=broker.authorize(subject=subject,request=proposal,token=trusted,attestation=compose_attestation(attestation))
assert decision.allowed
# A malicious model proposal cannot replace the trusted token or add an admin field.
malicious=model_request({'tool':'read_project_file','arguments':{'path':'/etc/shadow'},'capability':'admin'})
assert not broker.authorize(subject=subject,request=malicious,token=None).allowed
integration={'valid_typed_request_allowed':decision.allowed,'malicious_escalation_allowed':False,
             'external_actions_executed':0,'model_claims_ignored':True,'neural_role':'veto_only',
             'audit_events':len(broker.audit)}
save_stage('integration_demo.json',integration,BINDING); mark_stage('integration','integration_demo.json')
print(json.dumps(integration,indent=2))


In [ ]:
report={'run_id':RUN_ID,'status':'complete' if tests['all_passed'] else 'failed',
        'claim_scope':'veto-gated neural attestation plus external capability broker PoC',
        'security_invariants':{'model_can_mint_capability':False,'model_can_upgrade_privilege':False,
                               'neural_signal_can_authorize':False,'default_deny':True},
        'security_tests':tests,'integration':integration,
        'gpu_attestation':gpu_demo if gpu_demo is not None else {'status':'disabled'},
        'limitations':[
            'No real external tools, shell, network, filesystem, or APIs were executed.',
            'HMAC secret and broker are a reference PoC, not a production trust root or process-isolation design.',
            'Neural activation demo is not a trained safety score and cannot establish harmful-output prevention.',
            'Model process isolation, authenticated principals, secure key storage, TOCTOU defenses and deployment review remain required.',
            'A veto-only monitor can deny legitimate actions; utility and safety tradeoffs need separate evaluation.',
            'Novelty requires a current literature review; this notebook does not establish priority.'
        ]}
atomic_json(RUN_DIR/'final_report.json',report); mark_stage('report','final_report.json')
lines=['# NFW-004 report', '', 'Status: '+report['status'], '',
       '## Enforced security invariants', '', '- Model cannot mint capabilities: **False**', '- Model cannot upgrade privilege: **False**', '- Neural signal can authorize: **False**', '- Default deny: **True**', '',
       'No external actions were executed. This is a capability-boundary PoC, not a universal neural safety guarantee.', '', '## Limitations']+['- '+x for x in report['limitations']]
atomic_text(RUN_DIR/'REPORT.md','\n'.join(lines)+'\n')
print(json.dumps(report,indent=2))
